In [7]:
from pathlib import Path
import re
from dataclasses import dataclass
from typing import List, Optional, Dict
import pandas as pd

# ========= 設定 =========
# このノートブックと同じフォルダに RStudio_rireki.txt がある前提
HISTORY_PATH = Path("RStudio_rireki.txt")


# ========= データ構造 =========
@dataclass
class MiceCallInfo:
    line_no: int                      # RStudio_rireki.txt 内での行番号（1始まり）
    raw_line: str                     # mice() を含む元の1行
    lhs_var: Optional[str]            # mdata1 <- mice(...) の mdata1 部分
    first_arg: str                    # mice( ここが「補完前データ名」, ... )
    method_arg: Optional[str]         # method = ... の右辺（MD, "pmm" など）
    method_resolved: Optional[str]    # 実際のメソッド名（"cart" など）
    other_args: str                   # method 以外の引数文字列

    first_arg_source: Optional[str] = None      # 補完前データ変数の定義行
    first_arg_file: Optional[str] = None        # 読み込まれたファイル名（あれば）

    method_source: Optional[str] = None         # method が変数名だった場合の定義行

    completed_datasets: Optional[List[str]] = None  # complete(mdata1, ...) の出力変数名


# ========= 基本関数 =========
def read_history(path: Path) -> List[str]:
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return f.readlines()


def extract_parentheses_block(line: str, start_pos: int) -> str:
    """
    line 中の start_pos 以降で最初に現れる '(' から
    対応する ')' までを、入れ子を考慮して取り出す。
    例: 'mice(data, method=m, m=5)' -> '(data, method=m, m=5)'
    """
    depth = 0
    buf = []
    for ch in line[start_pos:]:
        buf.append(ch)
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
            if depth == 0:
                break
    return "".join(buf)


def parse_mice_call(line: str, line_no: int) -> Optional[MiceCallInfo]:
    """1行の中から mice(...) 呼び出しをパースして情報を抜き出す"""
    m_call = re.search(r"mice\s*\(", line)
    if not m_call:
        return None

    # 「?mice()」のようなヘルプ呼び出しは無視したいので簡易フィルタ
    if line.strip().startswith("?mice"):
        return None

    # 左辺 mdata1 <- mice(...) を取得
    lhs_var = None
    lhs_match = re.match(r"\s*([A-Za-z_][A-Za-z0-9_]*)\s*<-.*mice\s*\(", line)
    if lhs_match:
        lhs_var = lhs_match.group(1)

    # mice( ... ) 全体を取り出す
    call_str = line[m_call.start():]           # 'mice(...' から末尾まで
    block = extract_parentheses_block(call_str, call_str.find("("))
    args_str = block[1:-1]                     # 先頭 '(' と末尾 ')' を除く

    # トップレベルのカンマで引数を分割（入れ子の()は考慮）
    args_top = []
    current = []
    depth = 0
    for ch in args_str:
        if ch == "," and depth == 0:
            arg = "".join(current).strip()
            if arg:
                args_top.append(arg)
            current = []
        else:
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            current.append(ch)
    last = "".join(current).strip()
    if last:
        args_top.append(last)

    first_arg = args_top[0] if args_top else ""

    # method= の引数を取り出し、それ以外を other_args にまとめる
    method_arg = None
    other_parts = []
    for arg in args_top:
        if re.match(r"method\s*=", arg):
            method_arg = arg.split("=", 1)[1].strip()
        else:
            other_parts.append(arg)
    other_args = ", ".join(other_parts)

    return MiceCallInfo(
        line_no=line_no + 1,           # 1始まり
        raw_line=line.rstrip("\n"),
        lhs_var=lhs_var,
        first_arg=first_arg,
        method_arg=method_arg,
        method_resolved=None,
        other_args=other_args,
    )


def find_last_assignment_line(var: str, lines: List[str], line_no: int) -> Optional[int]:
    """
    mice() を呼び出している行より前で、
    var <- ... と代入されている最後の行番号を探す
    """
    pattern = re.compile(rf"\b{re.escape(var)}\s*<-[^=]")
    for i in range(line_no - 1, -1, -1):
        if pattern.search(lines[i]):
            return i
    return None


def extract_file_from_line(line: str) -> Optional[str]:
    """
    read.csv("xxx") や read.table("xxx") などから "xxx" を取り出す
    必要があれば readRDS 等も追加可能
    """
    m = re.search(r'read\w*\s*\(\s*["\']([^"\']+)["\']', line)
    if m:
        return m.group(1)
    return None


def resolve_method_name(method_arg: Optional[str],
                        method_source_line: Optional[str]) -> Optional[str]:
    """
    method 引数が文字列 or 変数名のときに、最終的な文字列（例: "cart"）を返す
    """
    if not method_arg:
        return None

    # すでに "cart" や 'pmm' などの文字列リテラルの場合
    m_str = re.match(r'["\']([^"\']+)["\']', method_arg.strip())
    if m_str:
        return m_str.group(1)

    # 変数名の場合、method_source_line から "xxx" を抜く
    if method_source_line:
        m = re.search(r'<-\s*["\']([^"\']+)["\']', method_source_line)
        if m:
            return m.group(1)

    return None


def find_complete_outputs(mice_var: Optional[str],
                          lines: List[str],
                          start_line: int) -> List[str]:
    """
    mice_result <- mice(...)
    の mice_result を第1引数に使った complete(mice_result, ...) の左辺変数名を全部拾う
    例: comp_data <- complete(mice_result, 1)
    """
    if not mice_var:
        return []
    pattern = re.compile(
        rf"\b([A-Za-z_][A-Za-z0-9_]*)\s*<-\s*complete\s*\(\s*{re.escape(mice_var)}\b"
    )
    outputs: List[str] = []
    for i in range(start_line, len(lines)):
        m = pattern.search(lines[i])
        if m:
            outputs.append(m.group(1))
    return outputs


def scan_mice_calls(path: Path) -> List[MiceCallInfo]:
    """RStudio_rireki.txt から mice() 関連の情報をすべて抽出"""
    lines = read_history(path)
    mice_calls: List[MiceCallInfo] = []

    for idx, line in enumerate(lines):
        if "mice(" not in line:
            continue
        info = parse_mice_call(line, idx)
        if not info:
            continue  # ?mice() などをスキップ

        # method が変数名っぽい場合、その定義行をさかのぼって取得
        if info.method_arg:
            # 変数名かどうかざっくり判定（英数字と _ のみ）
            if re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", info.method_arg.strip()):
                method_line_idx = find_last_assignment_line(info.method_arg.strip(), lines, idx)
                if method_line_idx is not None:
                    info.method_source = lines[method_line_idx].rstrip("\n")
            # 実際のメソッド名（"cart" など）に解決
        info.method_resolved = resolve_method_name(info.method_arg, info.method_source)

        # 補完前データ（最初の引数）が変数名っぽい場合、その定義行を取得
        if info.first_arg:
            if re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", info.first_arg.strip()):
                first_line_idx = find_last_assignment_line(info.first_arg.strip(), lines, idx)
                if first_line_idx is not None:
                    info.first_arg_source = lines[first_line_idx].rstrip("\n")
                    # さらに一段階さかのぼってファイル名を探す
                    # 例: testMatrix <- df1 / df1 <- read.csv("xxx.csv")
                    #    → first_arg=testMatrix, first_arg_source(testMatrix<-df1)
                    #      df1 の定義行を探す
                    m = re.search(r'<-\s*([A-Za-z_][A-Za-z0-9_]*)', info.first_arg_source)
                    if m:
                        parent_var = m.group(1)
                        parent_idx = find_last_assignment_line(parent_var, lines, first_line_idx)
                        if parent_idx is not None:
                            parent_line = lines[parent_idx].rstrip("\n")
                            file_path = extract_file_from_line(parent_line)
                            if file_path:
                                info.first_arg_file = file_path
                            else:
                                # 見つからなければ、とりあえずその行を残す
                                if info.first_arg_file is None:
                                    info.first_arg_file = parent_line
                # 直接 read.csv の可能性も一応チェック
                if info.first_arg_file is None and info.first_arg_source:
                    fp = extract_file_from_line(info.first_arg_source)
                    if fp:
                        info.first_arg_file = fp

        # mice 結果（lhs_var）を使った complete(...) の出力名を探す
        info.completed_datasets = find_complete_outputs(info.lhs_var, lines, idx + 1)

        mice_calls.append(info)

    return mice_calls


def build_summary_dataframe(mice_calls: List[MiceCallInfo]) -> pd.DataFrame:
    """抽出結果を pandas.DataFrame にまとめる"""
    rows = []
    for i, info in enumerate(mice_calls, start=1):
        rows.append(
            {
                "order": i,
                "line_no": info.line_no,
                "mice_object": info.lhs_var or "",
                "pre_imputation_data": info.first_arg,
                "pre_data_assignment": info.first_arg_source or "",
                "pre_data_file_or_source": info.first_arg_file or "",
                "method_arg_raw": info.method_arg or "",
                "method_resolved": info.method_resolved or "",
                "other_args": info.other_args,
                "completed_data_names": ";".join(info.completed_datasets or []),
                "raw_line": info.raw_line,
            }
        )
    df = pd.DataFrame(rows)
    return df


# ========= ここから実行部 =========
if not HISTORY_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {HISTORY_PATH}")

print(f"[INFO] Reading history file: {HISTORY_PATH}")
mice_infos = scan_mice_calls(HISTORY_PATH)

if not mice_infos:
    print("No actual mice() calls found.")
else:
    df_summary = build_summary_dataframe(mice_infos)

    # Jupyter 上で表示
    display(df_summary)

    # 同じフォルダに CSV 保存
    out_csv = HISTORY_PATH.with_name("mice_imputation_summary.csv")
    df_summary.to_csv(out_csv, index=False)
    print(f"\n[INFO] Summary CSV saved to:\n  {out_csv}")


[INFO] Reading history file: RStudio_rireki.txt


,order,line_no,mice_object,pre_imputation_data,pre_data_assignment,pre_data_file_or_source,method_arg_raw,method_resolved,other_args,completed_data_names,raw_line
0,1,4897,,testMatrix,1647238234300:testMatrix<-df1,1647238222832:df1<-scale(UsedP),MD,cart,"testMatrix, m=M, maxit=MAXIT",,"1647238235731:mdata1<-mice(testMatrix,method=M..."
1,2,4919,,testMatrix,1647238234300:testMatrix<-df1,1647238222832:df1<-scale(UsedP),MD,cart,"testMatrix, m=maxData[1], maxit=maxData[2]",,"1647238514252:mdata2<-mice(testMatrix,method=M..."
2,3,4940,,fdata,1647238516714:fdata<-scale(UsedP),,MD,cart,"fdata, m=maxData[1], maxit=maxData[2]",,"1647238516718:mdata2<-mice(fdata,method=MD,m=m..."
3,4,11273,,,,,,,,,1766038344198:?mice()
4,5,11281,,,,,,,,,1766038380000:?mice()



[INFO] Summary CSV saved to:
  mice_imputation_summary.csv


In [4]:
!python /home/is/i-yasuaki/scan_mice_calls_mice_imputation.py \
"/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal_regression/Imputation_Search/RStudio_rireki.txt"


/share/script/hosttype: Command not found.
/share/lib/skel/env: Command not found.
python: can't open file '/home/is/i-yasuaki/scan_mice_calls_mice_imputation.py': [Errno 2] No such file or directory
